# 35 - Generar fixtures reales para `real_baseline`

Objetivo del notebook:

1. Tomar un ejemplo sagital y uno axial desde Google Drive.
2. Aplicar un preprocesamiento compatible con los modelos finales.
3. Guardar archivos `.npy` de prueba para que Codex/Claude pueda validar `real_baseline`.
4. Generar un `.zip` con:
   - `sagittal_sample_input.npy`
   - `axial_sample_input.npy`
   - versiones `CHW`
   - previews `.png`
   - `fixture_summary.json`

> Importante: esto sirve para validar carga, shape, inferencia real y flujo técnico.  
> No representa validación clínica ni calidad poblacional del modelo.

In [ ]:
# ============================================
# 0) Montar Google Drive
# ============================================
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================
# 1) Configuración principal
# ============================================
from pathlib import Path
import os, json, hashlib, math, shutil, zipfile
import numpy as np

DRIVE_ROOT = Path("/content/drive/MyDrive")
PFI_ROOT = DRIVE_ROOT / "PFI_MVP"

SAGITTAL_CKPT = PFI_ROOT / "models/final/sagittal_spider_multiclass_final_best.pt"
AXIAL_CKPT    = PFI_ROOT / "models/final/axial_t2_alkafri_final_best.pt"

OUT_DIR = PFI_ROOT / "fixtures/real_baseline"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Recomendado:
# - Si ya sabés qué archivo usar, completá estas rutas.
# - Si las dejás en None, el notebook va a listar candidatos para que elijas.
SAG_SOURCE_PATH = None
AXIAL_SOURCE_PATH = None

# Ejemplos posibles:
# SAG_SOURCE_PATH = "/content/drive/MyDrive/PFI_MVP/datasets/SPIDER/.../case001.mha"
# AXIAL_SOURCE_PATH = "/content/drive/MyDrive/PFI_MVP/datasets/AlKafri/.../slice001.dcm"
# AXIAL_SOURCE_PATH = "/content/drive/MyDrive/PFI_MVP/datasets/AlKafri/.../dicom_series_folder"

print("PFI_ROOT:", PFI_ROOT)
print("Output:", OUT_DIR)
print("Sagittal checkpoint exists:", SAGITTAL_CKPT.exists(), SAGITTAL_CKPT)
print("Axial checkpoint exists:", AXIAL_CKPT.exists(), AXIAL_CKPT)

In [ ]:
# ============================================
# 2) Imports y utilidades
# ============================================
import warnings
warnings.filterwarnings("ignore")

try:
    import torch
    print("torch:", torch.__version__)
except Exception as e:
    raise RuntimeError("Falta torch en este entorno. En Colab normalmente ya está disponible.") from e

try:
    from PIL import Image
    PIL_AVAILABLE = True
except Exception:
    PIL_AVAILABLE = False

try:
    import pydicom
    PYDICOM_AVAILABLE = True
except Exception:
    PYDICOM_AVAILABLE = False
    print("pydicom no instalado. Si necesitás DICOM, ejecutá: !pip install pydicom")

try:
    import SimpleITK as sitk
    SITK_AVAILABLE = True
except Exception:
    SITK_AVAILABLE = False
    print("SimpleITK no instalado. Si necesitás MHA/MHD, ejecutá: !pip install SimpleITK")

try:
    import nibabel as nib
    NIB_AVAILABLE = True
except Exception:
    NIB_AVAILABLE = False
    print("nibabel no instalado. Si necesitás NIfTI, ejecutá: !pip install nibabel")


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def torch_load(path: Path):
    # weights_only=False porque algunos checkpoints guardan metadata además del state_dict.
    # Solo usar con artifacts propios/confiables.
    return torch.load(str(path), map_location="cpu", weights_only=False)


def checkpoint_state_dict(checkpoint):
    if isinstance(checkpoint, dict):
        for key in ["model_state_dict", "state_dict", "model"]:
            if key in checkpoint and isinstance(checkpoint[key], dict):
                return checkpoint[key]
        if checkpoint and all(torch.is_tensor(v) for v in checkpoint.values()):
            return checkpoint
    raise ValueError("No se pudo detectar state_dict en el checkpoint.")


def strip_prefixes(state_dict):
    cleaned = {}
    for k, v in state_dict.items():
        nk = k
        for prefix in ["module.", "model."]:
            if nk.startswith(prefix):
                nk = nk[len(prefix):]
        cleaned[nk] = v
    return cleaned


def infer_num_classes_from_state_dict(state_dict):
    for key in ["out_conv.weight", "out.weight"]:
        if key in state_dict:
            return int(state_dict[key].shape[0])
    candidates = [(k, v) for k, v in state_dict.items() if torch.is_tensor(v) and v.ndim == 4]
    if not candidates:
        return None
    return int(candidates[-1][1].shape[0])


def infer_base_channels_from_state_dict(state_dict, plane):
    if plane == "sagittal":
        for key in ["enc1.block.0.weight", "enc1.0.weight", "enc1.net.0.weight"]:
            if key in state_dict:
                return int(state_dict[key].shape[0])
    if plane == "axial":
        for key in ["e1.net.0.weight", "e1.block.0.weight", "e1.0.weight"]:
            if key in state_dict:
                return int(state_dict[key].shape[0])
    for k, v in state_dict.items():
        if torch.is_tensor(v) and v.ndim == 4:
            return int(v.shape[0])
    return None


def summarize_checkpoint(path: Path, plane: str):
    ckpt = torch_load(path)
    sd = strip_prefixes(checkpoint_state_dict(ckpt))
    summary = {
        "path": str(path),
        "exists": path.exists(),
        "size_mb": round(path.stat().st_size / (1024*1024), 2) if path.exists() else None,
        "sha256": sha256_file(path) if path.exists() else None,
        "checkpoint_type": type(ckpt).__name__,
        "checkpoint_keys": list(ckpt.keys())[:30] if isinstance(ckpt, dict) else None,
        "state_dict_keys_sample": list(sd.keys())[:15],
        "num_classes_ckpt": int(ckpt.get("num_classes")) if isinstance(ckpt, dict) and "num_classes" in ckpt else None,
        "num_classes_inferred": infer_num_classes_from_state_dict(sd),
        "base_channels_ckpt": int(ckpt.get("base_channels")) if isinstance(ckpt, dict) and "base_channels" in ckpt else None,
        "base_channels_inferred": infer_base_channels_from_state_dict(sd, plane),
        "target_size_ckpt": ckpt.get("target_size") if isinstance(ckpt, dict) and "target_size" in ckpt else None,
        "sagittal_axis_ckpt": ckpt.get("sagittal_axis") if isinstance(ckpt, dict) and "sagittal_axis" in ckpt else None,
        "slice_strategy_ckpt": ckpt.get("slice_strategy") if isinstance(ckpt, dict) and "slice_strategy" in ckpt else None,
    }
    return ckpt, sd, summary

sag_ckpt, sag_sd, sag_summary = summarize_checkpoint(SAGITTAL_CKPT, "sagittal")
ax_ckpt, ax_sd, ax_summary = summarize_checkpoint(AXIAL_CKPT, "axial")

print("Sagittal checkpoint summary:")
print(json.dumps({k:v for k,v in sag_summary.items() if k not in ["sha256"]}, indent=2, default=str))

print("
Axial checkpoint summary:")
print(json.dumps({k:v for k,v in ax_summary.items() if k not in ["sha256"]}, indent=2, default=str))

In [ ]:
# ============================================
# 3) Resolver target_size y parámetros esperados
# ============================================
def as_hw(value, default=(256, 256)):
    if value is None:
        return default
    if isinstance(value, torch.Tensor):
        value = value.detach().cpu().tolist()
    if isinstance(value, (list, tuple)) and len(value) == 2:
        return (int(value[0]), int(value[1]))
    if isinstance(value, int):
        return (int(value), int(value))
    return default

SAG_TARGET = as_hw(sag_summary.get("target_size_ckpt"), default=(256, 256))
AX_TARGET  = as_hw(ax_summary.get("target_size_ckpt"), default=(256, 256))

SAG_AXIS = sag_summary.get("sagittal_axis_ckpt")
if SAG_AXIS is None:
    SAG_AXIS = 2
SAG_AXIS = int(SAG_AXIS)

print("SAG_TARGET:", SAG_TARGET)
print("AX_TARGET:", AX_TARGET)
print("SAG_AXIS:", SAG_AXIS)
print("Axial num_classes:", ax_summary["num_classes_ckpt"] or ax_summary["num_classes_inferred"])
print("Sagittal num_classes:", sag_summary["num_classes_ckpt"] or sag_summary["num_classes_inferred"])

In [ ]:
# ============================================
# 4) Lectura de imágenes/volúmenes + preprocesamiento
# ============================================
def read_npy_or_npz(path: Path):
    obj = np.load(str(path), allow_pickle=False)
    if isinstance(obj, np.lib.npyio.NpzFile):
        keys = list(obj.keys())
        for pref in ["image", "img", "arr_0", "volume", "data"]:
            if pref in keys:
                return np.asarray(obj[pref])
        return np.asarray(obj[keys[0]])
    return np.asarray(obj)


def read_image_2d(path: Path):
    if not PIL_AVAILABLE:
        raise RuntimeError("PIL no disponible.")
    img = Image.open(str(path)).convert("F")
    return np.asarray(img, dtype=np.float32)


def read_mha_mhd(path: Path):
    if not SITK_AVAILABLE:
        raise RuntimeError("SimpleITK no instalado. Ejecutá: !pip install SimpleITK")
    img = sitk.ReadImage(str(path))
    arr = sitk.GetArrayFromImage(img)  # z,y,x
    return np.asarray(arr, dtype=np.float32)


def read_nifti(path: Path):
    if not NIB_AVAILABLE:
        raise RuntimeError("nibabel no instalado. Ejecutá: !pip install nibabel")
    obj = nib.load(str(path))
    return np.asarray(obj.get_fdata(), dtype=np.float32)


def dicom_sort_key(path: Path):
    if not PYDICOM_AVAILABLE:
        return str(path)
    try:
        ds = pydicom.dcmread(str(path), stop_before_pixels=True, force=True)
        if hasattr(ds, "InstanceNumber"):
            return (0, int(ds.InstanceNumber))
        if hasattr(ds, "ImagePositionPatient"):
            return (1, float(ds.ImagePositionPatient[-1]))
    except Exception:
        pass
    return (9, str(path))


def read_dicom_file(path: Path):
    if not PYDICOM_AVAILABLE:
        raise RuntimeError("pydicom no instalado. Ejecutá: !pip install pydicom")
    ds = pydicom.dcmread(str(path), force=True)
    arr = ds.pixel_array.astype(np.float32)
    slope = float(getattr(ds, "RescaleSlope", 1.0))
    intercept = float(getattr(ds, "RescaleIntercept", 0.0))
    return arr * slope + intercept


def read_dicom_dir(path: Path):
    files = []
    for p in path.rglob("*"):
        if p.is_file() and p.suffix.lower() in [".dcm", ".dicom", ""]:
            files.append(p)
    files = sorted(files, key=dicom_sort_key)
    if not files:
        raise ValueError(f"No encontré DICOMs en carpeta: {path}")
    mid = len(files) // 2
    print(f"DICOM dir: {len(files)} archivos. Usando archivo central:", files[mid])
    return read_dicom_file(files[mid])


def read_any(path_like):
    path = Path(path_like)
    if not path.exists():
        raise FileNotFoundError(path)
    if path.is_dir():
        return read_dicom_dir(path)
    name = path.name.lower()
    suf = path.suffix.lower()
    if suf in [".npy", ".npz"]:
        return read_npy_or_npz(path)
    if name.endswith(".nii.gz") or suf == ".nii":
        return read_nifti(path)
    if suf in [".mha", ".mhd"]:
        return read_mha_mhd(path)
    if suf in [".dcm", ".dicom"]:
        return read_dicom_file(path)
    if suf in [".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"]:
        return read_image_2d(path)
    raise ValueError(f"Formato no soportado o no reconocido: {path}")


def robust_normalize(arr, p_low=1, p_high=99):
    arr = np.asarray(arr, dtype=np.float32)
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    lo, hi = np.percentile(arr, [p_low, p_high])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        mn, mx = float(np.min(arr)), float(np.max(arr))
        if mx <= mn:
            return np.zeros_like(arr, dtype=np.float32)
        lo, hi = mn, mx
    out = np.clip(arr, lo, hi)
    return ((out - lo) / (hi - lo)).astype(np.float32)


def resize_2d(arr, target_hw):
    h, w = target_hw
    img = Image.fromarray(np.asarray(arr, dtype=np.float32))
    img = img.resize((w, h), resample=Image.BILINEAR)
    return np.asarray(img, dtype=np.float32)


def extract_sagittal_slice(volume, axis=2):
    arr = np.squeeze(np.asarray(volume, dtype=np.float32))
    if arr.ndim == 2:
        return arr
    if arr.ndim != 3:
        raise ValueError(f"Esperaba 2D o 3D para sagital. Shape recibido: {arr.shape}")
    axis = int(axis)
    if axis < 0 or axis >= arr.ndim:
        axis = int(np.argmin(arr.shape))
    idx = arr.shape[axis] // 2
    sl = np.take(arr, idx, axis=axis)
    print(f"Sagital: volumen {arr.shape}, axis={axis}, sliceIndex={idx}, slice shape={sl.shape}")
    return sl


def extract_axial_image(arr):
    arr = np.squeeze(np.asarray(arr, dtype=np.float32))
    if arr.ndim == 2:
        return arr
    if arr.ndim == 3:
        idx = arr.shape[0] // 2
        print(f"Axial: stack {arr.shape}, sliceIndex={idx}, slice shape={arr[idx].shape}")
        return arr[idx]
    raise ValueError(f"Esperaba 2D o 3D para axial. Shape recibido: {arr.shape}")


def prepare_model_input_2d(source_arr, target_hw, plane, sagittal_axis=2):
    arr2d = extract_sagittal_slice(source_arr, sagittal_axis) if plane == "sagittal" else extract_axial_image(source_arr)
    arr2d = robust_normalize(arr2d)
    arr2d = resize_2d(arr2d, target_hw)
    return np.clip(arr2d, 0, 1).astype(np.float32)


def save_preview(arr2d, path: Path):
    img = Image.fromarray((np.clip(arr2d, 0, 1) * 255).astype(np.uint8))
    img.save(str(path))

In [ ]:
# ============================================
# 5) Buscar candidatos si no cargaste rutas manuales
# ============================================
SUPPORTED_EXTS = [".npy", ".npz", ".mha", ".mhd", ".nii", ".gz", ".dcm", ".dicom", ".png", ".jpg", ".jpeg", ".tif", ".tiff"]

def find_candidates(root: Path, keywords=None, limit=80):
    keywords = keywords or []
    out = []
    skip_parts = {"models", "results", "fixtures", ".git", "__pycache__"}
    for p in root.rglob("*"):
        if len(out) >= limit:
            break
        if not p.is_file():
            continue
        parts_lower = {part.lower() for part in p.parts}
        if any(s in parts_lower for s in skip_parts):
            continue
        name = p.name.lower()
        if not any(name.endswith(ext) for ext in SUPPORTED_EXTS):
            continue
        full = str(p).lower()
        score = 0
        for kw in keywords:
            if kw.lower() in full:
                score += 10
        if any(m in full for m in ["mask", "label", "seg", "annotation"]):
            score -= 5
        if any(x in full for x in ["image", "img", "scan", "t2", "sag", "ax"]):
            score += 3
        out.append((score, p))
    return [p for _, p in sorted(out, key=lambda x: (-x[0], str(x[1])))[:limit]]

if SAG_SOURCE_PATH is None:
    print("No definiste SAG_SOURCE_PATH. Candidatos posibles para sagital:")
    for i, p in enumerate(find_candidates(PFI_ROOT, keywords=["spider", "sag"], limit=40)):
        print(f"{i:02d}: {p}")

if AXIAL_SOURCE_PATH is None:
    print("
No definiste AXIAL_SOURCE_PATH. Candidatos posibles para axial:")
    for i, p in enumerate(find_candidates(PFI_ROOT, keywords=["alkafri", "sudirman", "axial", "t2"], limit=40)):
        print(f"{i:02d}: {p}")

print("
Si la lista muestra archivos correctos, copiá una ruta en SAG_SOURCE_PATH / AXIAL_SOURCE_PATH en la celda 1 y rerun desde ahí.")

In [ ]:
# ============================================
# 6) OPCIÓN RÁPIDA: usar variables ya existentes de otro notebook
# ============================================
# Usá esta celda solo si venís de un notebook donde ya existen sag_input y axial_input.
# Si no existen, no hace nada.

if "sag_input" in globals() and "axial_input" in globals():
    print("Detecté variables sag_input y axial_input en memoria.")
    sag_arr = np.squeeze(np.asarray(sag_input, dtype=np.float32))
    ax_arr = np.squeeze(np.asarray(axial_input, dtype=np.float32))
    sag_prepared = resize_2d(robust_normalize(sag_arr), SAG_TARGET)
    ax_prepared = resize_2d(robust_normalize(ax_arr), AX_TARGET)
    SAG_SOURCE_PATH = "variable:sag_input"
    AXIAL_SOURCE_PATH = "variable:axial_input"
    sag_raw = sag_arr
    ax_raw = ax_arr
    print("sagital shape:", sag_prepared.shape)
    print("axial shape:", ax_prepared.shape)
else:
    print("No existen sag_input/axial_input en memoria. Usá rutas SAG_SOURCE_PATH y AXIAL_SOURCE_PATH.")

In [ ]:
# ============================================
# 7) Generar fixtures desde rutas, si no se usaron variables en memoria
# ============================================
if "sag_prepared" not in globals() or "ax_prepared" not in globals():
    if SAG_SOURCE_PATH is None or AXIAL_SOURCE_PATH is None:
        raise RuntimeError(
            "Falta definir SAG_SOURCE_PATH y/o AXIAL_SOURCE_PATH en la celda 1. "
            "Elegí rutas desde los candidatos impresos, o usá variables sag_input/axial_input."
        )

    SAG_SOURCE_PATH = Path(SAG_SOURCE_PATH)
    AXIAL_SOURCE_PATH = Path(AXIAL_SOURCE_PATH)

    sag_raw = read_any(SAG_SOURCE_PATH)
    ax_raw = read_any(AXIAL_SOURCE_PATH)

    sag_prepared = prepare_model_input_2d(sag_raw, SAG_TARGET, plane="sagittal", sagittal_axis=SAG_AXIS)
    ax_prepared  = prepare_model_input_2d(ax_raw, AX_TARGET, plane="axial")

print("sagital source shape:", np.asarray(sag_raw).shape)
print("sagital prepared shape:", sag_prepared.shape, sag_prepared.dtype, float(sag_prepared.min()), float(sag_prepared.max()))
print("axial source shape:", np.asarray(ax_raw).shape)
print("axial prepared shape:", ax_prepared.shape, ax_prepared.dtype, float(ax_prepared.min()), float(ax_prepared.max()))

In [ ]:
# ============================================
# 8) Guardar .npy, previews y summary
# ============================================
sag_path = OUT_DIR / "sagittal_sample_input.npy"
ax_path = OUT_DIR / "axial_sample_input.npy"
sag_chw_path = OUT_DIR / "sagittal_sample_input_chw.npy"
ax_chw_path = OUT_DIR / "axial_sample_input_chw.npy"
sag_png = OUT_DIR / "sagittal_sample_input_preview.png"
ax_png = OUT_DIR / "axial_sample_input_preview.png"

np.save(sag_path, sag_prepared.astype(np.float32))
np.save(ax_path, ax_prepared.astype(np.float32))
np.save(sag_chw_path, sag_prepared[None, :, :].astype(np.float32))
np.save(ax_chw_path, ax_prepared[None, :, :].astype(np.float32))

save_preview(sag_prepared, sag_png)
save_preview(ax_prepared, ax_png)

summary = {
    "purpose": "Fixtures deidentificados para validar carga e inferencia real_baseline en AI Module.",
    "warning": "No constituyen validacion clinica ni dataset de evaluacion poblacional.",
    "output_dir": str(OUT_DIR),
    "sagittal": {
        "source_path": str(SAG_SOURCE_PATH),
        "source_shape": list(np.asarray(sag_raw).shape),
        "prepared_shape": list(sag_prepared.shape),
        "target": list(SAG_TARGET),
        "sagittal_axis": SAG_AXIS,
        "file": str(sag_path),
        "file_sha256": sha256_file(sag_path),
        "chw_file": str(sag_chw_path),
        "chw_file_sha256": sha256_file(sag_chw_path),
        "preview": str(sag_png),
        "checkpoint": sag_summary,
    },
    "axial": {
        "source_path": str(AXIAL_SOURCE_PATH),
        "source_shape": list(np.asarray(ax_raw).shape),
        "prepared_shape": list(ax_prepared.shape),
        "target": list(AX_TARGET),
        "file": str(ax_path),
        "file_sha256": sha256_file(ax_path),
        "chw_file": str(ax_chw_path),
        "chw_file_sha256": sha256_file(ax_chw_path),
        "preview": str(ax_png),
        "checkpoint": ax_summary,
    },
}

summary_path = OUT_DIR / "fixture_summary.json"
summary_path.write_text(json.dumps(summary, indent=2, default=str), encoding="utf-8")

print("Archivos guardados:")
for p in [sag_path, ax_path, sag_chw_path, ax_chw_path, sag_png, ax_png, summary_path]:
    print("-", p, "| exists:", p.exists())

print("
Shape para pasar a Claude:")
print("sagital shape:", sag_prepared.shape)
print("axial shape:", ax_prepared.shape)
print("axial target_size confirmado:", AX_TARGET)

In [ ]:
# ============================================
# 9) Smoke-test directo contra checkpoints finales, si el repo AI Module está disponible
# ============================================
import sys
AI_REPO_CANDIDATES = [
    Path("/content/PFI_MVPTest_Enzo_AImodule"),
    PFI_ROOT / "PFI_MVPTest_Enzo_AImodule",
    DRIVE_ROOT / "PFI_MVPTest_Enzo_AImodule",
]

ai_service_path = None
for repo in AI_REPO_CANDIDATES:
    candidate = repo / "ai_service"
    if candidate.exists():
        ai_service_path = candidate
        break

if ai_service_path is None:
    print("No encontré el repo AI Module en Colab. Smoke-test de modelo salteado.")
    print("Esto está OK si solo necesitabas los .npy para pasárselos a Claude/Codex.")
else:
    print("Usando AI service path:", ai_service_path)
    sys.path.insert(0, str(ai_service_path))
    from pfi_ai_service.model_architectures import build_checkpoint_model

    sag_model = build_checkpoint_model("sagittal_spider", sag_ckpt)
    ax_model = build_checkpoint_model("axial_t2_alkafri", ax_ckpt)
    sag_model.eval()
    ax_model.eval()

    with torch.inference_mode():
        sag_tensor = torch.from_numpy(sag_prepared[None, None, :, :].astype(np.float32))
        ax_tensor = torch.from_numpy(ax_prepared[None, None, :, :].astype(np.float32))
        sag_out = sag_model(sag_tensor)
        ax_out = ax_model(ax_tensor)

    print("sag model output:", tuple(sag_out.shape))
    print("axial model output:", tuple(ax_out.shape))
    print("sag model training:", sag_model.training)
    print("axial model training:", ax_model.training)

In [ ]:
# ============================================
# 10) Comprimir para descargar / subir al repo
# ============================================
zip_path = OUT_DIR / "real_baseline_fixtures.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in [sag_path, ax_path, sag_chw_path, ax_chw_path, sag_png, ax_png, summary_path]:
        z.write(p, arcname=p.name)

print("ZIP:", zip_path)
print("ZIP sha256:", sha256_file(zip_path))
print("ZIP size MB:", round(zip_path.stat().st_size / (1024*1024), 3))

try:
    from google.colab import files
    # Descomentar si querés descargar automáticamente:
    # files.download(str(zip_path))
except Exception:
    pass

## Mensaje sugerido para Claude/Codex

Copiá y pegá esto cuando el notebook termine:

```text
Generé los fixtures reales para continuar AI-007/AI-008.

Archivos:
- sagittal_sample_input.npy
- axial_sample_input.npy
- sagittal_sample_input_chw.npy
- axial_sample_input_chw.npy
- fixture_summary.json

Shapes:
- sagital shape: <pegar salida>
- axial shape: <pegar salida>
- axial target_size confirmado: <pegar salida>

Aclaración:
Son fixtures deidentificados de dataset público y sirven para validar carga/inferencia real_baseline.
No representan validación clínica ni calidad final.
```